In [1]:
from pathlib import Path

import numpy as np
import pyvista as pv
from dolfinx.io import XDMFFile
from eikonax import derivator, preprocessing, solver, tensorfield
from ls_prior import builder
from mpi4py import MPI

from cardiac_electrophysiology import fibertensor, parameter, prior, ptsmap
from cardiac_electrophysiology.ls_bip import components, logging, posterior, utilities

In [2]:
data_path = Path("../data/patient_01/")
basis_vecs_path = data_path / "basis_vecs.npy"
vtu_mesh_path = data_path / "mesh_with_fibers_tags.vtu"
xdmf_mesh_path = data_path / "mesh_with_fibers_tags.xdmf"
log_file_path = Path("posterior_logfile.log")

basis_vectors = np.load(basis_vecs_path)
pv_mesh = pv.read(vtu_mesh_path)
with XDMFFile(MPI.COMM_WORLD, xdmf_mesh_path, "r") as xdmf:
    dlx_mesh = xdmf.read_mesh(name="Grid")
vertices = pv_mesh.points
simplices = pv_mesh.cells.reshape(-1, 4)[:, 1:4]

In [3]:
fiber_vectors = np.array(pv_mesh.cell_data["fibers"])
fiber_transformator = parameter.AngleFiberTransformator(
    basis_vectors[..., 0], basis_vectors[..., 1]
)
fiber_angles = fiber_transformator.compute_angle_from_fiber(fiber_vectors)
mean_fiber_angle = np.mean(fiber_angles)
angle_transformator = parameter.AngleParameterTransformator(mean_fiber_angle)
ground_truth_parameter = angle_transformator.compute_parameter_from_angle(fiber_angles)
num_simplices = simplices.shape[0]
mean_parameter_vector = angle_transformator.mean_parameter * np.ones(num_simplices)
mean_angle_vector = mean_fiber_angle * np.ones(num_simplices)
longitudinal_velocity_vector = np.full(num_simplices, 3)
transversal_velocity_vector = np.full(num_simplices, 1)

In [4]:
initial_site_ind = 12650
noise_variance = 1e-3
num_observations = 1000

logger_settings = logging.LoggerSettings(
    do_printing=False,
    logfile_path=log_file_path,
)

prior_settings = builder.BilaplacianPriorSettings(
    mesh=dlx_mesh,
    mean_vector=mean_parameter_vector,
    kappa=1,
    tau=1,
    seed=0,
)
fiber_tensor_settings = fibertensor.FiberTensorSettings(
    dimension=3,
    mean_angle_vector=mean_angle_vector,
    basis_vectors_one=basis_vectors[..., 0],
    basis_vectors_two=basis_vectors[..., 1],
    longitudinal_velocities=longitudinal_velocity_vector,
    transversal_velocities=transversal_velocity_vector,
)
eikonax_solver_settings = solver.SolverData(
    tolerance=1e-6,
    max_num_iterations=1000,
    max_value=1000,
    loop_type="jitted_while",
    use_soft_update=True,
    softminmax_order=20,
    softminmax_cutoff=0.01,
)
eikonax_derivator_settings = derivator.PartialDerivatorData(
    use_soft_update=True,
    softminmax_order=20,
    softminmax_cutoff=0.01,
)

In [5]:
prior_component = prior.FiberFieldPrior(prior_settings)
fiber_tensor = fibertensor.FiberTensor(fiber_tensor_settings)
tensor_field_mapping = tensorfield.LinearScalarMap()
tensor_field_object = tensorfield.TensorField(
    num_simplices=num_simplices,
    vector_to_simplices_map=tensor_field_mapping,
    simplex_tensor=fiber_tensor,
)

initial_sites = preprocessing.InitialSites(inds=(initial_site_ind,), values=(0,))
mesh_data = preprocessing.MeshData(vertices, simplices)
eikonax_solver = solver.Solver(mesh_data, eikonax_solver_settings, initial_sites)
eikonax_derivator = derivator.PartialDerivator(mesh_data, eikonax_derivator_settings, initial_sites)
eikonal_pts_map = ptsmap.EikonalPTSMap(eikonax_solver, eikonax_derivator, tensor_field_object)

In [6]:
ground_truth_solution = eikonal_pts_map.evaluate_forward(ground_truth_parameter)
rng = np.random.default_rng(seed=0)
noise = rng.normal(loc=0.0, scale=np.sqrt(noise_variance), size=vertices.shape[0])
noisy_solution = ground_truth_solution + noise
observation_inds = rng.integers(low=0, high=vertices.shape[0], size=num_observations)
observations = noisy_solution[observation_inds]

In [7]:
precision_values = np.full(num_observations, 1 / noise_variance)
observation_matrix = utilities.assemble_vertex_observation_matrix(
    vertices.shape[0], observation_inds
)
noise_precision_matrix = utilities.assemble_diagonal_precision_matrix(precision_values)

log_likelihood = components.GaussianLogLikelihood(
    observations, observation_matrix, noise_precision_matrix
)
posterior_logger = logging.LSBIPLogger(logger_settings)
log_posterior = posterior.LogPosterior(
    log_likelihood, eikonal_pts_map, prior_component, posterior_logger
)

In [8]:
cost = log_posterior.evaluate_cost(mean_parameter_vector)
gradient = log_posterior.evaluate_gradient(mean_parameter_vector)

In [9]:
cost = log_posterior.evaluate_cost(ground_truth_parameter)
gradient = log_posterior.evaluate_gradient(ground_truth_parameter)